# RAG with Databricks Vector Search

This notebook replaces `wk2 - langchain/4.langchain.rag.ipynb` with the fully Databricks-native stack:

| Component | wk2 (OpenAI shim) | wk2-databricks (native) |
|---|---|---|
| LLM | `ChatOpenAI` → Databricks endpoint | `ChatDatabricks` |
| Embeddings | `OpenAIEmbeddings` → Databricks endpoint | `DatabricksEmbeddings` |
| Vector Store | `SKLearnVectorStore` (in-memory) | `DatabricksVectorSearch` (managed, persistent) |

## Prerequisites

Complete **once** in your Databricks workspace before running this notebook:

1. Create a **Vector Search endpoint**  
   *Catalog → Vector Search → Create endpoint*  
   Note the endpoint name and set it as `DATABRICKS_VS_ENDPOINT` in `.env`.

2. Create a **Unity Catalog schema** (run in a Databricks SQL warehouse or notebook):  
   ```sql
   CREATE CATALOG IF NOT EXISTS cs4603;
   CREATE SCHEMA  IF NOT EXISTS cs4603.rag;
   ```

3. Set `.env` values:  
   ```
   DATABRICKS_VS_ENDPOINT="<your-endpoint-name>"
   DATABRICKS_VS_INDEX="cs4603.rag.docs_index"
   ```

4. The first run of **Cell 4 (Create Index)** will provision the index — this takes ~1–2 minutes.

## Cell 1 — Imports & Bootstrap

In [2]:
%run databricks_native_common.py

print(f"LLM endpoint   : {DATABRICKS_MODEL}")
print(f"VS endpoint    : {vs_config.endpoint_name}")
print(f"VS index       : {vs_config.index_name}")

Connected to: https://adb-2907158998162760.0.azuredatabricks.net/
Model endpoint: databricks-qwen35-122b-a10b
Vector Search index: cs4603.rag.docs_index


NameError: name 'DATABRICKS_MODEL' is not defined

## Cell 2 — Load Documents

Using `WebBaseLoader` to fetch two LangGraph documentation pages.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://langchain-ai.github.io/langgraph/concepts/agentic_concepts/",
    "https://langchain-ai.github.io/langgraph/concepts/low_level/",
]

docs_list = []
for url in urls:
    loader = WebBaseLoader(
        url,
        header_template={"User-Agent": "Mozilla/5.0 (LangChain RAG demo)"},
    )
    docs_list.extend(loader.load())

print(f"Loaded {len(docs_list)} document(s)")
print(f"Source: {docs_list[0].metadata.get('source')}")

## Cell 3 — Split Documents

Uses `RecursiveCharacterTextSplitter` with tiktoken — the standard chunking approach.
Each chunk is ≤250 tokens, matching the `databricks-gte-large-en` context window.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=25,   # small overlap so context isn't lost at boundaries
)
doc_splits = text_splitter.split_documents(docs_list)

print(f"Split into {len(doc_splits)} chunks")
print(f"\nSample chunk:\n{doc_splits[0].page_content[:300]}")

## Cell 4 — Create Databricks Vector Search Index

Uses a **Direct Access Index**: you manage chunking and embedding, Databricks manages storage and ANN search.

- `embedding_dimension=1024` matches `databricks-gte-large-en`
- Run this cell **once** to provision the index. Re-running raises a `ResourceAlreadyExistsException` which is caught and ignored.

In [ ]:
from databricks.ai_search.client import VectorSearchClient

vsc = VectorSearchClient(
    workspace_url=DATABRICKS_HOST,
    personal_access_token=DATABRICKS_TOKEN,
    disable_notice=True,
)

try:
    vsc.create_direct_access_index(
        endpoint_name=vs_config.endpoint_name,
        index_name=vs_config.index_name,
        primary_key="id",
        embedding_dimension=1024,          # gte-large-en output size
        embedding_vector_column="embedding",
        schema={
            "id":        "string",
            "text":      "string",
            "source":    "string",
            "embedding": "array<float>",
        },
    )
    print(f"Index '{vs_config.index_name}' created successfully.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index '{vs_config.index_name}' already exists — skipping creation.")
    else:
        raise

## Cell 5 — Connect LangChain to the Index

`DatabricksVectorSearch` wraps the index as a LangChain `VectorStore`.
It uses `embeddings` (DatabricksEmbeddings) to embed queries at retrieval time.

In [ ]:
from databricks_langchain.vectorstores import DatabricksVectorSearch

vectorstore = DatabricksVectorSearch(
    index_name=vs_config.index_name,
    embedding=embeddings,
    text_column="text",
    columns=["text", "source"],   # metadata columns to return with results
)

print("Connected to vector store:", vs_config.index_name)

## Cell 6 — Embed and Upsert Documents

`add_texts()` embeds each chunk via `DatabricksEmbeddings` and upserts into the index.

This is safe to re-run — existing IDs are updated in place.

In [ ]:
import uuid

texts     = [d.page_content for d in doc_splits]
metadatas = [{"source": d.metadata.get("source", "")} for d in doc_splits]
ids       = [str(uuid.uuid4()) for _ in doc_splits]

added_ids = vectorstore.add_texts(texts=texts, metadatas=metadatas, ids=ids)

print(f"Upserted {len(added_ids)} chunks into '{vs_config.index_name}'")

## Cell 7 — Similarity Search (sanity check)

Verify the index is working before building the full chain.

In [ ]:
results = vectorstore.similarity_search("What is agent memory?", k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} (source: {doc.metadata.get('source', 'n/a')}) ---")
    print(doc.page_content[:300])

## Cell 8 — RAG Chain (LCEL pipeline)

Parallel branches:
- `RunnablePassthrough()` keeps the original question for the prompt
- `retriever | format_docs` fetches relevant chunks and formats them as a single string

Both feed into the prompt → LLM → parser.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant that answers questions using only the provided context. "
     "If the answer is not in the context, say you don't know."),
    ("human",
     "Context:\n{context}\n\nQuestion: {question}"),
])

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context" : retriever | RunnableLambda(format_docs),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

## Cell 9 — Ask questions

In [ ]:
questions = [
    "What is agent memory in LangGraph?",
    "How does LangGraph handle state between nodes?",
    "What is the difference between a graph node and an edge?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_chain.invoke(q)}")

## Cell 10 — Inspect retrieved sources

Helper to see *which* chunks influenced the answer.

In [ ]:
def ask_with_sources(question: str):
    docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)
    sources = list({d.metadata.get("source", "unknown") for d in docs})
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"Sources: {sources}")

ask_with_sources("What types of memory does an agent have?")